<a href="https://www.kaggle.com/code/mrafraim/dl-day-47-inference-pipeline?scriptVersionId=309360642" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Day 47: Inference Pipeline
*Strict Pipeline: Image → Preprocessing → Model → Prediction*

Welcome to Day 47!

What You’ll Learn Today:

1. Clean separation of training vs inference
2. Deterministic pipeline design (no randomness)
3. Exact preprocessing replication
4. Writing a reusable prediction function
5. Debugging real-world inference failures

If you found this notebook helpful, your **<b style="color:skyblue;">UPVOTE</b>** would be greatly appreciated! It helps others discover the work and supports continuous improvement.

---

# Hard Separation Rule

Most real-world ML bugs come from
👉 mixing training logic into inference

This causes:
- inconsistent outputs
- non-reproducible predictions
- silent accuracy drops

### Forbidden in Inference

- optimizer
- loss function
- backward()
- training loop
- data augmentation


### Allowed

- model loading
- preprocessing
- forward pass
- prediction logic

# End-to-End Inference Flow

```markdown
Raw Image (file)
↓
Load Image (PIL)
↓
Preprocessing (same as training)
↓
Tensor (C × H × W)
↓
Add batch dimension (1 × C × H × W)
↓
Model (eval mode)
↓
Output logits
↓
Convert to prediction
↓
Return label
```

# Preprocessing (STRICT REPLICATION)

Your model expects data in a specific distribution.

If preprocessing differs even slightly
→ model breaks silently

### Must Match Training:

✔ image size  
✔ grayscale / RGB  
✔ normalization mean/std  
✔ tensor format  

**Why?**

Model learned weights based on THIS exact input distribution.

In [1]:
import torchvision.transforms as transforms

# EXACT SAME as training
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # ensure correct channel
    transforms.Resize((28, 28)),                 # match model input size
    transforms.ToTensor(),                       # convert to tensor
    transforms.Normalize((0.5,), (0.5,))         # same normalization
])

# Load Model Properly

Model weights alone are useless without:
- correct architecture
- correct device mapping
- eval mode

### Hidden Failure Mode

If architecture mismatches:

→ weights load incorrectly<br>
→ predictions become garbag

```python

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Recreate EXACT architecture
model = CNN().to(device)

# Load trained weights
state_dict = torch.load("best_model.pth", map_location=device)
model.load_state_dict(state_dict)

# Switch to inference mode
model.eval()
```

**Why `model.eval()` is NON-NEGOTIABLE**

It changes behavior of:

- Dropout → OFF
- BatchNorm → uses learned statistics

Without this
→ predictions become inconsistent

# Core Prediction Function

A good inference function is:

✔ self-contained  
✔ deterministic  
✔ minimal  
✔ reusable  

### Input:
- image path

### Output:
- predicted class index

```python
from PIL import Image

def predict_image(image_path, model, transform, device):
    
    # Step 1: Load image
    image = Image.open(image_path)

    # Step 2: Apply preprocessing
    image = transform(image)

    # Step 3: Add batch dimension
    image = image.unsqueeze(0).to(device)

    # Step 4: Inference (no gradients)
    with torch.no_grad():
        outputs = model(image)
        pred = outputs.argmax(dim=1)

    return pred.item()

```

# Test the Pipeline

Before deployment
→ verify pipeline works end-to-end

Otherwise
→ bugs surface in production

```python
pred_class = predict_image("sample.png", model, transform, device)

print("Predicted class index:", pred_class)
```

# Map Prediction to Label

Model outputs
→ integer index (not human-friendly)

We map it to labels.

```python
classes = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("Prediction:", classes[pred_class])

```

# Common Silent Bugs

1. Missing normalization
→ model sees wrong scale

2. Wrong image size
→ shape mismatch or poor prediction

3. No batch dimension
→ runtime error

4. model.train() instead of eval()
→ randomness in output

5. Different preprocessing than training
→ completely wrong predictions


> 90% deployment failures come from THIS, not the model.

# Determinism (Production Requirement)

Same input → SAME output

Always.

### Why?

- debugging
- monitoring
- user trust

### What Breaks Determinism?

- dropout active
- random augmentations
- inconsistent preprocessing

# Production-Ready Wrapper

This is what real systems expose.

```python
def predict(image_path):

    model.eval()

    image = Image.open(image_path)
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(image)
        pred = output.argmax(dim=1).item()

    return classes[pred]
```

# Key Takeaways from Day 47
- Inference is a separate system, not a continuation of training
- Preprocessing consistency is the #1 failure point
- `model.eval()` is mandatory
- Prediction must be deterministic
- Clean function design = production readiness

---

<p style="text-align:center; color:skyblue; font-size:18px;">
© 2026 Mostafizur Rahman
</p>
